# Preparations

If you want to prepare for the workshop in advance, here is what you can do. This is only
relevant if you plan to run the notebooks **locally** (if you are using Colab, you can skip
it entirely).

Doing this beforehand means you can start on the exercises straight away instead of waiting
for downloads.

## 1. Install the environment

Use [uv](https://docs.astral.sh/uv/) (preferred) or
[conda](https://docs.conda.io/projects/conda/en/latest/user-guide/getting-started.html).

**1.1 Install uv** (once, if you do not already have it):

```bash
# macOS / Linux
curl -LsSf https://astral.sh/uv/install.sh | sh

# Windows (PowerShell)
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

**1.2 Set up the environment.** Run this in a terminal, from inside the folder containing
this repository:

```bash
uv sync    # creates the environment with all the necessary packages
```

## 2. Check the packages

In [ ]:
import pandas as pd
import numpy as np
import transformers
import torch
import umap
import sklearn
import datasets

print("All libraries imported successfully.")
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("UMAP:", umap.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)

## 3. Preload the model and data

Running the two cells below downloads everything ahead of time, so you are not waiting on
the conference wifi.

First the language model. Any of the three works: they differ only in how much disk space
and memory they take. `ModernBERT-base` is the one the workshop uses (though, if you want 
to experement, or have limited space, feel free to download larger or smaller model).

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM

model_id = "answerdotai/ModernBERT-base"
# model_id = "answerdotai/ModernBERT-large"  # alternative (LARGER) model
# model_id = "bert-base-uncased"             # alternative (SMALLER) model

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForMaskedLM.from_pretrained(model_id).eval()

print(f"Loaded {model_id}")
print(f"  {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"  {len(tokenizer):,} tokens in the vocabulary")

And now the data — three sets of true and false statements, used in notebook 3.

In [ ]:
from datasets import load_dataset

ds = load_dataset("carlomarxx/trilemma-of-truth", "city_locations")
assert ds["train"].shape == (3999, 11)
print("Loaded city_locations dataset")

ds = load_dataset("carlomarxx/trilemma-of-truth", "med_indications")
assert ds["train"].shape == (3849, 11)
print("Loaded med_indications dataset")

ds = load_dataset("carlomarxx/trilemma-of-truth", "word_definitions")
assert ds["train"].shape == (4717, 11)
print("Loaded word_definitions dataset")


## 4. The event model and the workshop data

In the  second half of the workshop we focus on:
1. the model that is trained for the synthetic health record
2. Dataset of 50,000-(synthetic) patients generated via [Synthea](https://synthetichealth.github.io/synthea/)

Together they are about **14 MB**, so this part is quick.

Both are committed to this repository. If you cloned it you already have them and the cell
below will simply find them; if you did not, the same cell fetches them from the Hugging Face
Hub instead.

In [ ]:
import pathlib

MODEL_REPO = "carlomarxx/synthea-bert"
DATA_REPO = "carlomarxx/synthea-workshop-data"


def resolve(local_path, repo_id, repo_type="model"):
    """Prefer the copy in this repository; fall back to the Hub if it is not here."""
    if pathlib.Path(local_path).exists():
        return str(local_path), "already in this repository"
    from huggingface_hub import snapshot_download

    return snapshot_download(repo_id, repo_type=repo_type), f"downloaded from {repo_id}"


MODEL, model_source = resolve("../models/synthea-bert", MODEL_REPO)
DATA, data_source = resolve("../data/derived/workshop", DATA_REPO, "dataset")

print(f"event model:   {MODEL}\n               ({model_source})")
print(f"workshop data: {DATA}\n               ({data_source})")

Now load it, rather than just downloading it. The cell below rebuilds the architecture and loads the weights into it, which is what actually fails if something is wrong.

In [ ]:
import sys

sys.path.insert(0, "../scripts")

from event_bert import EventBertForMaskedLM

vocabulary = pd.read_csv(f"{DATA}/vocabulary.csv")
event_model = EventBertForMaskedLM.from_pretrained(
    MODEL, expected_vocab_size=len(vocabulary)
)

print(f"Loaded the event model")
print(f"  {sum(p.numel() for p in event_model.parameters()):,} parameters")
print(f"  {len(vocabulary)} tokens in the event vocabulary")
print(
    f"  {(vocabulary.kind == 'event').sum()} event types, "
    f"{(vocabulary.kind == 'background').sum()} background, "
    f"{(vocabulary.kind == 'special').sum()} special"
)

## 5. That's it

If every cell above ran without error, you are ready.

Everything the workshop downloads now sits in your local caches: the ~600 MB language model
for the first half, and the ~14 MB event model and data extract for the second. None of it
will need the conference wifi on the day.